# Linguistic Features Training Notebook Ver1

🔹 Goal: Fine-tune a RoBERTa model on SemEval-2024 Task 8 Subtask A with linguistic features from spaCy library.

🔹 Output: The linguistic model ver1 saved as "linguistic_model_ver1.pth".

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Setup & Install Dependencies

In [ ]:
%pip install transformers jsonlines tqdm
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 66.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizer, RobertaModel
import spacy
import jsonlines
import numpy as np
import joblib
from collections import Counter
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

## Step 2: Load spaCy & Define Linguistic Feature Extraction

In [ ]:
# Load spaCy's English model
nlp = spacy.load("en_core_web_md")

# Define global sets
FUNCTION_WORDS = {'AUX', 'CCONJ', 'DET', 'INTJ', 'PART', 'PRON', 'ADP', 'SCONJ'}
CONTENT_WORDS = {'NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'}

# Predefine patterns and POS/DEP tags based on dataset
ALL_DEP_RELATIONS = nlp.pipe_labels['parser']
ALL_POS_TAGS = nlp.pipe_labels['tagger']

def extract_linguistic_features(text):
    doc = nlp(text)

    # 1. Dependency Relation Frequency (normalized)
    dep_counter = Counter([token.dep_ for token in doc])
    dep_freq = np.array([dep_counter.get(dep, 0) for dep in ALL_DEP_RELATIONS])
    dep_freq = dep_freq / (np.sum(dep_freq) + 1e-8)

    # 2. Tree Depth: Max token.depth
    tree_depth = max([token.head.i - token.i for token in doc if token.head != token]) if len(doc) > 1 else 0
    tree_depth = np.array([tree_depth])

    # 3. Specific syntactic pattern presence (e.g., nsubj -> ROOT -> dobj)
    pattern_present = any(
        token.dep_ == 'ROOT' and
        any(child.dep_ == 'nsubj' for child in token.children) and
        any(child.dep_ == 'dobj' for child in token.children)
        for token in doc
    )
    pattern_feature = np.array([1 if pattern_present else 0])

    # 4. POS tag bigram frequency (simplified)
    pos_bigrams = Counter()
    for i in range(len(doc) - 1):
        bigram = (doc[i].pos_, doc[i+1].pos_)
        pos_bigrams[bigram] += 1
    # Use a few common patterns for simplicity
    common_bigrams = [('NOUN', 'VERB'), ('VERB', 'NOUN'), ('ADJ', 'NOUN')]
    pos_bigram_feats = np.array([pos_bigrams.get(bigram, 0) for bigram in common_bigrams])

    # 5. Function word to content word ratio
    function_count = sum(1 for token in doc if token.pos_ in FUNCTION_WORDS)
    content_count = sum(1 for token in doc if token.pos_ in CONTENT_WORDS)
    func_content_ratio = np.array([function_count / (content_count + 1e-8)])

    # 6. Number of clauses per sentence (using "ccomp", "advcl", etc.)
    clause_tags = {"ccomp", "xcomp", "advcl", "relcl", "acl"}
    clause_count = sum(1 for token in doc if token.dep_ in clause_tags)
    clause_feat = np.array([clause_count])

    # 7. Head POS Tag Distribution
    head_pos_counter = Counter([token.head.pos_ for token in doc])
    common_heads = ['NOUN', 'VERB', 'ADJ']
    head_pos_feats = np.array([head_pos_counter.get(pos, 0) for pos in common_heads])

    # Combine all features
    features = np.concatenate([
        dep_freq,
        tree_depth,
        pattern_feature,
        pos_bigram_feats,
        func_content_ratio,
        clause_feat,
        head_pos_feats
    ])

    return features

## Step 3: Load Dataset & Preprocess

### 🔹 3.1 Prepare and Load the Scaler

In [ ]:
# Create a Scaler for the first time run
# scaler = StandardScaler()

# Save the Scaler
# joblib.dump(scaler, "/content/drive/MyDrive/SemEval-2024 Task 8/models/scaler.pkl")

# Load scaler if needed
scaler = joblib.load("/content/drive/MyDrive/SemEval-2024 Task 8/models/scaler.pkl")

### 🔹 3.2 Define Custom Text Dataset Class

In [ ]:
class CustomTextDataset(Dataset):
    def __init__(self, file_path, tokenizer, scaler, max_length=512):
        self.data = []
        self.max_length = max_length
        self.scaler = scaler

        with jsonlines.open(file_path) as reader:
            lines = list(reader)  # So tqdm knows total length
            for obj in tqdm(lines, desc=f"Processing {file_path}"):
                text = obj["text"]
                label = obj["label"]

                encoding = tokenizer(
                    text,
                    truncation=True,
                    padding='max_length',
                    max_length=max_length,
                    return_tensors='pt'
                )

                ling_feats = extract_linguistic_features(text)
                ling_feats = self.scaler.fit_transform(ling_feats.reshape(1, -1))

                self.data.append((encoding, ling_feats, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        encoding, ling_feats, label = self.data[idx]
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'ling_feats': torch.tensor(ling_feats, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

## Step 4: Define the Model

In [ ]:
class LinguisticRobertaClassifier(nn.Module):
    def __init__(self, ling_feat_dim, num_labels=2):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.roberta.config.hidden_size + ling_feat_dim, num_labels)  # concat CLS + linguistic features

    # def forward(self, input_ids, attention_mask, ling_feats):
    #     outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
    #     cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token
    #     combined = torch.cat((cls_emb, ling_feats), dim=1)
    #     combined = self.dropout(combined)
    #     logits = self.classifier(combined)
    #     return logits

    def forward(self, input_ids, attention_mask, ling_feats):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # [batch_size, 768]

        ling_feats = ling_feats.view(ling_feats.size(0), -1)  # 🧠 Reshape to [batch_size, ling_feat_dim]

        combined = torch.cat((cls_emb, ling_feats), dim=1)  # [batch_size, 768 + ling_feat_dim]
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        return logits

## Step 5: Prepare for Training

### 🔹 5.1 Create Train & Dev Datasets

In [ ]:
# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Define paths
train_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_train_monolingual.jsonl"
dev_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_dev_monolingual.jsonl"

# Create datasets for the first time run
train_dataset = CustomTextDataset(train_file, tokenizer, scaler)
dev_dataset = CustomTextDataset(dev_file, tokenizer, scaler)

ling_feat_dim = len(extract_linguistic_features("Example text"))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Processing /content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_train_monolingual.jsonl: 100%|██████████| 119757/119757 [2:07:05<00:00, 15.71it/s]
Processing /content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_dev_monolingual.jsonl: 100%|██████████| 5000/5000 [04:19<00:00, 19.25it/s]


### 🔹 5.2 Save the Train and Dev Datasets

In [ ]:
# Save the datasets for the first time run
torch.save(train_dataset, "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/train_dataset_ver1.pt")
torch.save(dev_dataset, "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/dev_dataset_ver1.pt")

### 🔹 5.3 Load Train & Dev Datasets

In [ ]:
train_dataset = torch.load(
    "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/train_dataset_ver1.pt",
    weights_only=False
)
dev_dataset = torch.load(
    "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/dev_dataset_ver1.pt",
    weights_only=False
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False)

In [ ]:
ling_feat_dim = len(extract_linguistic_features("Example text"))
print(ling_feat_dim)

55


### 🔹 5.4 Initialize Model

In [ ]:
model = LinguisticRobertaClassifier(ling_feat_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LinguisticRobertaClassifier(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              

### 🔹 5.5 Define Optimizer & Loss Function

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

## Step 6: Train the Model

### 🔹 6.1 Define Training Function

In [ ]:
def train_and_validate(model, train_loader, val_loader, optimizer, criterion, device="cuda", epochs=3):
    model.to(device)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        # Training phase
        model.train()
        total_train_loss = 0

        for batch in tqdm(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ling_feats = batch['ling_feats'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, ling_feats=ling_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        print(f"→ Avg Train Loss: {avg_train_loss:.4f}")

        # Validation phase
        model.eval()
        total_val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for batch in tqdm(val_loader):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                ling_feats = batch['ling_feats'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, ling_feats=ling_feats)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        avg_val_loss = total_val_loss / len(val_loader)
        macro_f1 = f1_score(val_labels, val_preds, average='macro')
        print(f"→ Avg Dev Loss: {avg_val_loss:.4f}, Macro F1: {macro_f1:.4f}")

### 🔹 6.2 Train the Model

In [ ]:
train_and_validate(model, train_loader, dev_loader, optimizer, criterion, epochs=3)


Epoch 1/3


100%|██████████| 7485/7485 [37:51<00:00,  3.29it/s]


→ Avg Train Loss: 0.0232


100%|██████████| 313/313 [00:30<00:00, 10.43it/s]


→ Avg Dev Loss: 1.8577, Macro F1: 0.6923

Epoch 2/3


...

→ Avg Dev Loss: 3.3900, Macro F1: 0.5238

Epoch 3/3


100%|██████████| 7485/7485 [37:51<00:00,  3.30it/s]


→ Avg Train Loss: 0.0062


100%|██████████| 313/313 [00:29<00:00, 10.45it/s]

→ Avg Dev Loss: 2.7180, Macro F1: 0.6017


## Step 7: Save the Trained Model

In [ ]:
# Set directory for saving linguistic model
linguistic_model_path = "/content/drive/MyDrive/SemEval-2024 Task 8/models/linguistic models"
os.makedirs(linguistic_model_path, exist_ok=True)

# Save model checkpoint
model_save_path = os.path.join(linguistic_model_path, "linguistic_model_ver1.pth")
torch.save(model.state_dict(), model_save_path)

print(f"Linguistic model saved at: {model_save_path}")

Linguistic model saved at: /content/drive/MyDrive/SemEval-2024 Task 8/models/linguistic models/linguistic_model_ver1.pth
